# precio_operado vs precio de mercado. 
### Unifircar tablas y calcular desvio 

In [0]:
from pyspark.sql.functions import *

In [0]:
df_tx = spark.table("silver_byma.tipo_cambio")
df_tx = df_tx.withColumnRenamed("simbolo_titulo", "simbolo")

df_prices = spark.table("silver_byma.precios_historicos")


df_analysis = df_tx.join(
    df_prices,
    on=["simbolo", "fecha"],
    how="left"
)

In [0]:
df_analysis = df_analysis.withColumn(
    "desvio",
    col("precio_operado_usd") - col("precio_usd")
).withColumn(
    "desvio_pct",
    (col("precio_operado_usd") - col("precio_usd")) / col("precio_usd")
)

In [0]:
df_analysis = df_analysis.select(
    "id_transaccion",
    "id_cliente",
    "simbolo",
    "fecha",
    "tipoTran",
    "cantidad",
    "precio_operado_usd",
    "precio_usd",
    "desvio",
    "desvio_pct"
)

In [0]:
df_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_byma.cotizaciones_historicas")